# DFT расчёт HEA L12 (32 атома) — CPU Версия

**Требование:**
1. `HEA_L12_SQS_32atoms.cif`
2. Псевдопотенциалы в папке `pseudo`

# Импорт

In [1]:
!apt-get update -qq
!apt-get install -y quantum-espresso > /dev/null
!pip install -q pymatgen

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 3.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.4/883.4 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.5/962.5 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 

In [ ]:
from pymatgen.core import Structure
import re

In [3]:
cif_file = "HEA_L12_SQS_32atoms.cif"
struct = Structure.from_file(cif_file)

pseudo_map = {
    "Al": "Al.pbesol-n-kjpaw_psl.1.0.0.UPF",
    "Ti": "Ti.upf",
    "Cr": "Cr.upf",
    "Fe": "Fe.pbesol-spn-kjpaw_psl.0.2.1.UPF",
    "Co": "Co.upf",
    "Ni": "Ni.upf"
}
masses = {"Al": 26.98, "Ti": 47.87, "Cr": 52.00, "Fe": 55.85, "Co": 58.93, "Ni": 58.69}

# Настройка QE

In [4]:
inp = f"""&CONTROL
    calculation = 'vc-relax'
    prefix = 'HEA_L12_32'
    pseudo_dir = './pseudo'
    outdir = './tmp'
    etot_conv_thr = 1.0d-4
    forc_conv_thr = 1.0d-3
    nstep = 200
    verbosity = 'high'
/
&SYSTEM
    ibrav = 0
    nat = {struct.num_sites}
    ntyp = {len(pseudo_map)}
    ecutwfc = 60
    ecutrho = 480
    occupations = 'smearing'
    smearing = 'methfessel-paxton'
    degauss = 0.02
    input_dft = 'pbesol'
/
&ELECTRONS
    conv_thr = 1.0d-8
    mixing_beta = 0.3
    electron_maxstep = 200
/
&IONS
    ion_dynamics = 'bfgs'
/
&CELL
    cell_dynamics = 'bfgs'
    press = 0.0
/
ATOMIC_SPECIES
"""

for el, fname in pseudo_map.items():
    inp += f" {el}  {masses[el]}  {fname}\n"

inp += "CELL_PARAMETERS angstrom\n"
for v in struct.lattice.matrix:
    inp += f" {v[0]:.6f} {v[1]:.6f} {v[2]:.6f}\n"

inp += "ATOMIC_POSITIONS crystal\n"
for site in struct:
    sym = site.species.elements[0].symbol
    fc = site.frac_coords
    inp += f" {sym}  {fc[0]:.6f} {fc[1]:.6f} {fc[2]:.6f}\n"

inp += "K_POINTS automatic\n 4 4 4  0 0 0\n"

with open("vc-relax.in", "w") as f:
    f.write(inp)

# Запуск

In [5]:
# Запуск
!pw.x < vc-relax.in 2>&1 | tee vc-relax.out

# Результат
with open("vc-relax.out", "r") as f:
    out = f.read()
energy = re.search(r'!\s+total energy\s+=\s+([\-\d\.]+)\s+Ry', out)
if energy:
    print(f"Энергия: {energy.group(1)} Ry")
else:
    print("Энергия не найдена")


     Program PWSCF v.6.7MaX starts on  5Apr2026 at 22:43:52 

     This program is part of the open-source Quantum ESPRESSO suite
     for quantum simulation of materials; please cite
         "P. Giannozzi et al., J. Phys.:Condens. Matter 21 395502 (2009);
         "P. Giannozzi et al., J. Phys.:Condens. Matter 29 465901 (2017);
          URL http://www.quantum-espresso.org", 
     in publications or presentations arising from this work. More details at
     http://www.quantum-espresso.org/quote

     Parallel version (MPI), running on     1 processors

     MPI processes distributed on     1 nodes
     Waiting for input...
     Reading input from standard input

     Current dimensions of program PWSCF are:
     Max number of different atomic species (ntypx) = 10
     Max number of k-points (npk) =  40000
     Max angular momentum in pseudopotentials (lmaxx) =  3
     Message from routine read_pp_mesh:
     mismatch in mesh size, discarding the one in header
     Message from routin